# Hyperparameter Optimization with Bayesian Search Cross-Validation

In this notebook, we examine train several models, examine their performance, and create visualizations for these models related to explanations and their performance.
- Train models (XGBoost, Random Forest, kNN, AdaBoost, and Linear Regression)
- Examine test performance of each model using density scatter plots with RMSE/$R^2$/MBE information for all 4 models
- Use trained model to examine SHAP values on the test set for each model across datasets (bar plot and beeswarm plot)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats

from sklearnex import patch_sklearn
patch_sklearn()

import os
os.environ["SKLEARNEX_VERBOSE"] = "INFO"

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import statsmodels.api as sm
from sklearn.ensemble import AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical
import joblib
import mpl_scatter_density
from matplotlib.colors import LinearSegmentedColormap
import shap

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

Intel(R) Extension for Scikit-learn* enabled (https://github.com/intel/scikit-learn-intelex)


## Data

In [2]:
# Mapping dictionary
mapping_dict = {'Arid': 0, 'Snow': 1, 'Temperate': 2, 'Tropical': 3}

In [3]:
old_cols = ['BSA', 'NDBI', 'NDVI', 'DEM']
new_cols = ['\u0394BSA', '\u0394NDBI', '\u0394NDVI', '\u0394DEM']

In [4]:
# NTL dataset
data_ntl                 = pd.read_csv('data/data_ntl.csv')
data_ntl['Coastal?']     = data_ntl['Coastal?'].astype(int)
data_ntl['Climate Zone'] = data_ntl['Climate Zone'].map(mapping_dict)
data_ntl.rename(columns=dict(zip(old_cols, new_cols)), inplace=True)

In [5]:
# SUE dataset
data_sue                 = pd.read_csv('data/data_sue.csv')
data_sue['Coastal?']     = data_sue['Coastal?'].fillna(False).astype(int)
data_sue['Climate Zone'] = data_sue['Climate Zone'].map(mapping_dict)
data_sue.rename(columns=dict(zip(old_cols, new_cols)), inplace=True)

In [6]:
features = ['\u0394BSA', '\u0394NDBI', '\u0394NDVI', '\u0394DEM', 'Coastal?', 'Climate Zone']
label    = 'SUHI'

## Helper Functions

In [7]:
def get_data(df, test_size=0.20):
    df = df[features+[label]].dropna().copy()
    X  = df[features]
    y  = df[label]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size,  random_state=1, shuffle=True)
    return X_train, X_test, y_train, y_test

- [Density Scatter Plot Code](https://stackoverflow.com/questions/20105364/how-can-i-make-a-scatter-plot-colored-by-density)
- [Density Scatter Plot Library](https://github.com/astrofrog/mpl-scatter-density/tree/main)

In [8]:
# Test Performance Graphs
def performance_graphs(model_ada, model_knn, model_ridge, X_test, y_test, dataset, filename):
    
    models = [model_ada, model_knn, model_ridge]
    if filename == 'SUE':
        titles = ['1(a) AdaBoost - SUE', '1(b) kNN - SUE', '1(c) Ridge Regression - SUE']
    if filename == 'NTL':
        titles = ['2(a) AdaBoost - NTL', '2(b) kNN - NTL', '2(c) Ridge Regression - NTL']
    
    # "Viridis-like" colormap with white background
    white_viridis = LinearSegmentedColormap.from_list('white_viridis', [
        (0, '#ffffff'),
        (1e-20, '#440053'),
        (0.2, '#404388'),
        (0.4, '#2a788e'),
        (0.6, '#21a784'),
        (0.8, '#78d151'),
        (1, '#fde624'),
    ], N=256)

    font = {'size': 14}
    plt.rc('font', **font)
    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.sans-serif"] = ["Verdana"]
    nrows, ncols = 1, len(models)
    fig = plt.figure(figsize=(15,5), constrained_layout=True, dpi=1000)
    
    lim = [-15,15]
    dpi=30
    for i in range(len(models)):
        y_pred = models[i].predict(X_test)
        rmse = mean_squared_error(y_test, y_pred, squared = False)
        r2   = r2_score(y_test, y_pred)
        mbe  = np.mean(y_pred - y_test)
        # Scatter Density Plot
        ax = fig.add_subplot(nrows, ncols, i+1, projection='scatter_density')
        density = ax.scatter_density(y_pred, y_test, cmap=white_viridis, dpi=dpi)
        ax.set_xlabel('Predicted {}'.format(label), fontsize=14)
        if i == 0:
            ax.set_ylabel('Observed {}'.format(label), fontsize=14)
        ax.set_title(titles[i], fontsize=18)
        # Line of Best Fit
        slope, intercept, r_value, p_value, std_err = stats.linregress(y_pred, y_test)
        y_test_scipy = slope * y_pred + intercept
        operator = "+" if intercept >= 0 else ""
        if round(abs(mbe), 2) == 0:
            perf_label = f"y = {slope:.2f}x{operator}${intercept:.2f}$; $R^2$={r_value ** 2:.2f}\nRMSE={rmse:.2f}\u00b0C; MBE$\\approx$0\u00b0C"
            if round(abs(intercept), 2) == 0:
                perf_label = f"y = {slope:.2f}x; $R^2$={r_value ** 2:.2f}\nRMSE={rmse:.2f}\u00b0C; MBE$\\approx$0\u00b0C"
        else:
            perf_label = f"y = {slope:.2f}x{operator}${intercept:.2f}$; $R^2$={r_value ** 2:.2f}\nRMSE={rmse:.2f}\u00b0C; MBE=${mbe:.2f}$\u00b0C"
            if round(abs(intercept), 2) == 0:
                perf_label = f"y = {slope:.2f}x; $R^2$={r_value ** 2:.2f}\nRMSE={rmse:.2f}\u00b0C; MBE=${mbe:.2f}$\u00b0C"
        ax.plot(y_pred, y_test_scipy, color='black', linestyle='solid', label=perf_label)
        ax.legend(loc='lower right')
        # 1:1 Line
        ax.plot(y_test, y_test, linestyle='dashed', color='red')
        ax.set_xlim(lim)
        ax.set_ylim(lim)
        ax.tick_params(axis='both', which='major', labelsize=14)
        
    plt.savefig('graphs/performance/supplemental_{}.svg'.format(filename), bbox_inches='tight', dpi=1000)

In [9]:
def run_search(dataset, filename, n_folds=10, n_iter = 20):
    X_train, X_test, y_train, y_test = get_data(dataset)
    
    # Define search space
    search_space_ada = {
        'n_estimators': Integer(10, 150),
        'loss': Categorical(['linear', 'square', 'exponential']),
        'learning_rate': Real(1e-6, 1e+6, prior='log-uniform')

    }
    
    # Create model
    model_ada = AdaBoostRegressor(random_state=1)

    # Perform Bayesian optimization with cross validation
    print('AdaBoost')
    bayes_search_ada = BayesSearchCV(estimator=model_ada, search_spaces=search_space_ada, n_iter=n_iter, cv=n_folds, n_jobs=-1, 
                                    error_score='raise', return_train_score=True, scoring = 'neg_root_mean_squared_error')
    bayes_search_ada.fit(X_train, y_train)
    
    # Print best score and parameters
    print(f"Best score AdaBoost: {bayes_search_ada.best_score_:.3f}")
    print(f"Best parameters AdaBoost: {bayes_search_ada.best_params_}")
    
    # Define search space for kNN
    search_space_knn = {
        'n_neighbors': Integer(1, 10), 
        'weights': Categorical(['uniform', 'distance'])
    }
    
    # Create model
    model_knn = KNeighborsRegressor()
    
    # Perform Bayesian optimization with cross validation
    print('\nkNN')
    bayes_search_knn = BayesSearchCV(estimator=model_knn, search_spaces=search_space_knn, n_iter=n_iter, cv=n_folds, n_jobs=-1, 
                                     error_score='raise', return_train_score=True, scoring = 'neg_root_mean_squared_error')
    bayes_search_knn.fit(X_train, y_train)

    # Print best score and parameters
    print(f"Best score kNN: {bayes_search_knn.best_score_:.3f}")
    print(f"Best parameters kNN: {bayes_search_knn.best_params_}")

    # Define search space for kNN
    search_space_ridge = {
        'alpha': Real(1e-6, 1e+6, prior='log-uniform')
    }
    
    # Create model
    model_ridge = Ridge()

    # Perform Bayesian optimization with cross validation
    print('\Ridge Regression')
    bayes_search_ridge = BayesSearchCV(estimator=model_ridge, search_spaces=search_space_ridge, n_iter=n_iter, cv=n_folds, n_jobs=-1, 
                                       error_score='raise', return_train_score=True, scoring = 'neg_root_mean_squared_error')
    bayes_search_ridge.fit(X_train, y_train)

    # Print best score and parameters
    print(f"Best score Ridge: {bayes_search_ridge.best_score_:.3f}")
    print(f"Best parameters Ridge: {bayes_search_ridge.best_params_}")
    
    performance_graphs(bayes_search_ada.best_estimator_, bayes_search_knn.best_estimator_, bayes_search_ridge.best_estimator_, X_test, y_test, dataset, filename)
        
    return bayes_search_ada.best_estimator_, bayes_search_knn.best_estimator_, bayes_search_ridge.best_estimator_, bayes_search_ada.cv_results_, bayes_search_knn.cv_results_, bayes_search_ridge.cv_results_


In [10]:
def save_cv_results(filename, cv_results_ada, cv_results_knn, cv_results_ridge):
    pd.DataFrame(cv_results_ada).to_csv('tables/cv_results/{}_ada.csv'.format(filename),index=False)
    pd.DataFrame(cv_results_knn).to_csv('tables/cv_results/{}_knn.csv'.format(filename),index=False)
    pd.DataFrame(cv_results_ridge).to_csv('tables/cv_results/{}_ridge.csv'.format(filename),index=False)
    return

In [11]:
def save_models(filename, best_ada, best_knn, best_ridge):
    joblib.dump(best_ada, 'models/cross_val/' + filename + '_ada_estimator.joblib') 
    joblib.dump(best_knn, 'models/cross_val/' + filename + '_knn_estimator.joblib') 
    joblib.dump(best_ridge, 'models/cross_val/' + filename + '_ridge_estimator.joblib')
    return

In [12]:
def run_all(dataset, filename):
    best_ada, best_knn, best_ridge, cv_results_ada, cv_results_knn, cv_results_ridge = run_search(dataset, filename)
        
    save_cv_results(filename, cv_results_ada, cv_results_knn, cv_results_ridge)
    save_models(filename, best_ada, best_knn, best_ridge)
    
    y_pred_ada  = best_ada.predict(dataset[features+[label]].dropna().copy()[features])
    y_pred_knn = best_knn.predict(dataset[features+[label]].dropna().copy()[features])
    y_pred_ridge = best_ridge.predict(dataset[features+[label]].dropna().copy()[features])
    y_full = dataset[features+[label]].dropna().copy()[label]
    
    print('Full Dataset Performance - {}'.format(filename))
    print('----------------------------')
    print('AdaBoost')
    print('R2: ', round(r2_score(y_full, y_pred_ada),2))
    print('Root Mean Squared Error: ', round(mean_squared_error(y_full, y_pred_ada, squared = False),2))
    print('Mean Bias Error: ', round(np.mean(y_pred_ada - y_full),2))
    print('----------------------------')
    print('kNN')
    print('R2: ', round(r2_score(y_full, y_pred_knn),2))
    print('Root Mean Squared Error: ', round(mean_squared_error(y_full, y_pred_knn, squared = False),2))
    print('Mean Bias Error: ', round(np.mean(y_pred_knn - y_full),2))
    print('----------------------------')
    print('Ridge Regression')
    print('R2: ', round(r2_score(y_full, y_pred_ridge),2))
    print('Root Mean Squared Error: ', round(mean_squared_error(y_full, y_pred_ridge, squared = False),2))
    print('Mean Bias Error: ', round(np.mean(y_pred_ridge - y_full),2))
    return

## Models

### SUE

In [13]:
run_all(data_sue, 'SUE')

AdaBoost
Best score AdaBoost: -1.954
Best parameters AdaBoost: OrderedDict([('learning_rate', 0.05492711144481306), ('loss', 'linear'), ('n_estimators', 88)])

kNN


The objective has been evaluated at point [10, 'distance'] before, using random point [8, 'distance']
The objective has been evaluated at point [9, 'distance'] before, using random point [4, 'uniform']
The objective has been evaluated at point [9, 'distance'] before, using random point [3, 'distance']
The objective has been evaluated at point [10, 'distance'] before, using random point [5, 'distance']


Best score kNN: -2.097
Best parameters kNN: OrderedDict([('n_neighbors', 9), ('weights', 'distance')])
\Ridge Regression
Best score Ridge: -2.030
Best parameters Ridge: OrderedDict([('alpha', 0.007490490746761028)])


All-NaN slice encountered
All-NaN slice encountered
All-NaN slice encountered
All-NaN slice encountered
All-NaN slice encountered
All-NaN slice encountered


Full Dataset Performance - SUE
----------------------------
AdaBoost
R2:  0.59
Root Mean Squared Error:  1.95
Mean Bias Error:  -0.12
----------------------------
kNN
R2:  0.9
Root Mean Squared Error:  0.95
Mean Bias Error:  0.01
----------------------------
Ridge Regression
R2:  0.56
Root Mean Squared Error:  2.03
Mean Bias Error:  -0.0


In [14]:
run_all(data_sue, 'SUE')

AdaBoost
Best score AdaBoost: -1.975
Best parameters AdaBoost: OrderedDict([('learning_rate', 0.1292741055707023), ('loss', 'linear'), ('n_estimators', 23)])

kNN


The objective has been evaluated at point [9, 'distance'] before, using random point [4, 'uniform']
The objective has been evaluated at point [9, 'distance'] before, using random point [2, 'uniform']
The objective has been evaluated at point [10, 'distance'] before, using random point [5, 'uniform']


Best score kNN: -2.097
Best parameters kNN: OrderedDict([('n_neighbors', 9), ('weights', 'distance')])
\Ridge Regression
Best score Ridge: -2.030
Best parameters Ridge: OrderedDict([('alpha', 0.008638927993374063)])


All-NaN slice encountered
All-NaN slice encountered
All-NaN slice encountered
All-NaN slice encountered
All-NaN slice encountered
All-NaN slice encountered


Full Dataset Performance - SUE
----------------------------
AdaBoost
R2:  0.58
Root Mean Squared Error:  1.98
Mean Bias Error:  -0.05
----------------------------
kNN
R2:  0.9
Root Mean Squared Error:  0.95
Mean Bias Error:  0.01
----------------------------
Ridge Regression
R2:  0.56
Root Mean Squared Error:  2.03
Mean Bias Error:  -0.0


### NTL

In [15]:
run_all(data_ntl, 'NTL')

AdaBoost
Best score AdaBoost: -2.534
Best parameters AdaBoost: OrderedDict([('learning_rate', 0.11233103878016071), ('loss', 'exponential'), ('n_estimators', 150)])

kNN
Best score kNN: -2.677
Best parameters kNN: OrderedDict([('n_neighbors', 9), ('weights', 'distance')])
\Ridge Regression
Best score Ridge: -2.664
Best parameters Ridge: OrderedDict([('alpha', 0.05948592069707159)])


All-NaN slice encountered
All-NaN slice encountered
All-NaN slice encountered
All-NaN slice encountered
All-NaN slice encountered
All-NaN slice encountered


Full Dataset Performance - NTL
----------------------------
AdaBoost
R2:  0.58
Root Mean Squared Error:  2.51
Mean Bias Error:  -0.45
----------------------------
kNN
R2:  0.91
Root Mean Squared Error:  1.17
Mean Bias Error:  0.01
----------------------------
Ridge Regression
R2:  0.54
Root Mean Squared Error:  2.65
Mean Bias Error:  -0.0
